<table>
  <tr>
    <td><div align="left"><font size="30">Temperature control lab</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120"></td>
  </tr>
</table>

**Modeling the Temperature Control Lab**

(c) Peter Corke 2026


# Introduction

[Temperature Control Lab](https://apmonitor.com/heat.htm)

This lab device has two heater/heatsink/sensor units.  Electrical power delivered to each unit is convereted to heat and dissipated by
convection to the ambient air and radiatively to the other unit.

<img src="https://apmonitor.com/pdc/uploads/Main/tclab_transparent.png" width="400">

# Modeling

| Quantity | Value |
| --- | --- |
| Initial temperature ($T_0$) | 296.15 K (23°C)
| Ambient temperature ($T_\infty$) | 296.15 K (23°C)
| Heater output ($Q_1$) | 0 to 1 W
| Heater factor ($\alpha_1$) | 0.01 W/(% heater)
| Heater output ($Q_2$) | 0 to 0.75 W
| Heater factor ($\alpha_2$) | 0.0075 W/(% heater)
| Heat capacity ($C_p$) | 500 J/kg-K
| Surface Area Not Between Heaters ($A$) | $1.0 \times 10^{-3} \text{ m}^2$ ($10 \text{ cm}^2$)
| Surface Area Between Heaters ($A_s$) | $2 \times 10^{-4} \text{ m}^2$ ($2 \text{ cm}^2$)
| Mass ($m$) | 0.004 kg (4 gm)
| Overall Heat Transfer Coefficient ($U$) | 10 W/m²-K
| Emissivity ($\epsilon$) | 0.9
| Stefan Boltzmann Constant ($\sigma$) | $5.67 \times 10^{-8} \text{ W/m}^2\text{-K}^4$

**Convective and Radiative Heat Transfer Equations:**

$$Q_{C12} = U A_s (T_2 - T_1)$$

$$Q_{R12} = \epsilon \sigma A (T_2^4 - T_1^4)$$


**Dynamic Temperature Response Equations:**

$$m c_p \frac{dT_1}{dt} = U A (T_\infty - T_1) + \epsilon \sigma A (T_\infty^4 - T_1^4) + Q_{C12} + Q_{R12} + \alpha_1 Q_1$$

$$m c_p \frac{dT_2}{dt} = U A (T_\infty - T_2) + \epsilon \sigma A (T_\infty^4 - T_2^4) - Q_{C12} - Q_{R12} + \alpha_2 Q_2$$

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

# parameters, compared to the table above, the heater factors α1/α2 are changed so that heater is a float [0,1], not a percentage [0,100]
T0 = 296.15       # Initial temperature (K)
T_inf = 296.15    # Ambient temperature (K)
Q1_max = 1.0      # Heater 1 maximum output (W)
α1 = 1            # Heater 1 factor (W/(heater))
Q2_max = 0.75     # Heater 2 maximum output (W)
α2 = 0.75         # Heater 2 factor (W/(heater))
Cp = 500.0        # Heat capacity (J/kg-K)
A = 1.0e-3        # Surface Area Not Between Heaters (m^2)
As = 2.0e-4       # Surface Area Between Heaters (m^2)
m = 0.004         # Mass (kg)
U = 10.0          # Overall Heat Transfer Coefficient (W/m^2-K)
ε = 0.9           # Emissivity
σ = 5.67e-8       # Stefan Boltzmann Constant (W/m^2-K^4)

In [ ]:
T1 = bd.INTEGRATOR(name="T1", x0=T0)  # Temperature of heater 1
T2 = bd.INTEGRATOR(name="T2", x0=T0)  # Temperature of heater 2


In [ ]:
Q1 = bd.STEP(T=10, on=Q1_max, name="heater1")   # Heater 1 input
Q2 = bd.STEP(T=5*60, on=Q2_max, name="heater2")  # Heater 2 input


In [ ]:

Q_C12 = U * As * (T2 - T1)  # Conduction heat transfer between heaters
Q_R12 = ε * σ * As * (T2**4 - T1**4)  # Radiation heat transfer between heaters

T1[0] = (U * A * (T_inf - T1) + ε * σ * A * (T_inf**4 - T1**4) + Q_C12 + Q_R12 + α1 * Q1) / (m * Cp)
T2[0] = (U * A * (T_inf - T2) + ε * σ * A * (T_inf**4 - T2**4) - Q_C12 - Q_R12 + α2 * Q2) / (m * Cp)


In [ ]:

T1C = T1 - 273.15  # Convert to Celsius
T2C = T2 - 273.15  # Convert to Celsius

bd.SCOPE(labels=("T_1", "T_2"),name="Temperature Scope", inputs=[T1C, T2C])  # display the temperatures on a scope

In [ ]:
bd.compile()  # check the diagram
print(bd)
# bd.report_summary()

In [ ]:
out = sim.run(bd, T=600, watch=[T1, T2])  # simulate for 120s
print(out)